<a href="https://colab.research.google.com/github/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud/blob/main/FaseA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase A: Gestión del Asistente

En esta primera fase del proyecto definiremos a grandes rasgos la estructura global del funcionamiento del asistente: las opciones que dará, la memoria a la que tendrá acceso, la estructura de esta última, etc. Nuestro principal objetivo será crear un menú claro y ordenado, lo más sencillo e intuitivo posible para que el asistente sea accesible y fácil de usar.

**Contenidos** <a id='toc0_'></a>   

[Librerías necesarias](#toc1)

[Esquema global del funcionamiento del asistente](#toc2)

[Interfaz del asistente](#toc3)

- [Saludo inicial](toc3_1)
  - [Bloque de memoria](toc3_1_1)
  - [Sin memoria previa: presentación y recabación de datos](toc3_1_2)
  - [Con memoria previa: saludo breve](toc3_1_3)

- [Menú de opciones](toc4)

- [Cohesión: función global](toc5)


## <a id='toc1'></a>[0. Librerías necesarias](#toc0_)


In [1]:
# Para administrar ARCHIVOS:
import json
import os
from google.colab import files

# Para usar EXPRESIONES REGULARES:
import re

# Para administrar IMÁGENES:
import PIL.Image

# Para grabar AUDIO:
!pip install openai-whisper -q
import whisper
from google.colab import output
from base64 import b64decode

# Para reproducir AUDIO:
!pip install edge-tts -q
import asyncio
import edge_tts
from IPython.display import Audio, display

# Para pasar de VOZ a AUDIO:
from whisper import load_model
model_whisper = load_model("small") # Cargamos el modelo Whisper

In [2]:
!pip install -q bitsandbytes
from transformers import BitsAndBytesConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
torch.cuda.empty_cache()

# Cargamos el modelo de Texto puro para MIRAR SI LO VAMOS A USAR O NO!!!!!
#id_model_texto = "Qwen/Qwen2.5-1.5B-Instruct"
id_model_texto = "Qwen/Qwen2.5-3B-Instruct"

# Configuración para ahorrar memoria (4 bits)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32
)


tokenizer_texto = AutoTokenizer.from_pretrained(id_model_texto)
model_texto = AutoModelForCausalLM.from_pretrained(
    id_model_texto,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Modelo de TEXTO cargado con éxito")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Modelo de TEXTO cargado con éxito


## <a id='toc2'></a>[1. Esquema global del funcionamiento del asistente](#toc0_)

Debido a la naturaleza de este mini-proyecto, vamos a enfocar el asistente en un único usuario, por lo que solo guardaremos información de dicho usuario. Cuando se empiece una consulta al asistente, este buscará en su "memoria" (datos proporcionados en consultas previas, que se guardaran en un archivo con formato _JSON_):
- si no encuentra este archivo, creará uno vacío, se presentará y le pedirá al usuario un nombre por el que poder dirigirse a él/ella. Después de recabar el nombre, procederá a ofrecerle los servicios que abarca (profundizado en Fase B).
- si encuentra el archivo, sabrá el nombre del usuario, por lo que directamente le saludará y ofrecerá distintas opciones.

A parte de las opciones de asistencia en el ámbito de salud (introducción de medicamentos en el registro, respuesta a preguntas sencillas, resumen del historial de medicinas y lectura en voz alta de documentos), también ofreceremos otras opciones técnicas pertinentes, como el cambio de nombre, la supresión parcial o total de la información recabada en el registro, etc. Sin embargo, para simplificar el esquema se podrá acceder a estas opciones mediante la opción de _Ajustes_.

Una vez realizada y finalizada la primera consulta, el asistente preguntará amablemente si se desea continuar hablando con el asistente o si, por el contrario, no desean consultar nada más. En el primer caso, el asistente volverá a ofrecer las opciones disponibles y, en el segundo, dará por finalizada la sesión y se cerrará la aplicación, guardándose toda la información de dicha sesión.

## <a id='toc3'></a>[2. Interfaz del asistente](#toc0_)


### <a id='toc3_1'></a>[2.1. Saludo inicial](#toc0_)



#### <a id='toc3_1_1'></a>[2.1.1. Bloque de memoria](#toc0_)

El primer paso para poder estructurar las interacciones asistente-usuario será crear un histórico con la información proporcionada. Si el archivo de memoria no existe, se entiende que estamos hablando por primera vez con el usuario.

Como la primera consulta y todas las posteriores se basarán en la información recabada en la memoria, es importante fijar una estructura clara y precisa para el documento. Por ello, hemos decidido que el archivo tendrá formato _JSON_ y contará con los siguientes campos:

- _nombre_ : apelativo con el que el asistente interpelará al usuario.
- _medicinas_ : lista de diccionarios que contendrá el nombre, dosis, frecuencia, fechas inicial y final de los medicamentos.
- _ultimas_adiciones_ : lista con los nombres de los últimos medicamentos agregados a la lista de _medicinas_ (este último apartado será útil a la hora de registrar nuevos medicamentos vía imágenes).

Pasando de la teoría a la práctica hemos escrito la siguiente función, que busca el archivo de memoria y, si no lo encuentra, crea uno vacío:

In [45]:
def cargar_memoria():
    if os.path.exists("memoria_salud.json"):
        with open("memoria_salud.json", "r") as f:
            return json.load(f)
    else:
        # Si no existe, creamos un perfil vacío
        perfil_vacio = {"nombre": None, "medicinas": [], "ultimas_adiciones": []}
        with open("memoria_salud.json", "w") as f:
          json.dump(perfil_vacio, f, indent=4)
        return perfil_vacio

In [46]:
def guardar_memoria(datos):
    with open("memoria_salud.json", "w") as f:
        json.dump(datos, f, indent=4)

In [62]:
# Inicializamos la memoria al empezar
memoria = cargar_memoria()
print(memoria)

{'nombre': 'Ana', 'medicinas': [], 'ultimas_adiciones': []}


#### <a id='toc3_1_2'>[2.1.2 Sin memoria previa: presentación y recabación de datos](#toc0_)

En este apartado abordamos el primer contacto entre el usuario y el asistente. Cuando nuestra función de carga determina que no existe un historial previo (es decir, el campo "nombre" se encuentra vacío o es _None_), el sistema asume que es la primera vez que interactúa con la persona. Por tanto, el objetivo aquí es programar una rutina de bienvenida donde el asistente se presente y solicite el nombre del usuario para poder ofrecerle un trato cercano y personalizado. Una vez recabado este dato mediante un input por pantalla, actualizaremos la memoria para que incluya el nombre de este usuario y así posteriormente poder guardar su historial médico.

Primero el programa solicita el nombre, lo registra en la variable de memoria y, lo más importante, llama a la función guardar_memoria().

Añadimos un modelo de escucha (`grabar_audio)`), un modelo de transcripción de audio (`model_whisper`, cargado en la primera sección) y un modelo para pasar el texto a voz (`generar_voz`). Con ayuda de estos tres modelos hemos escrito la función `presentacion`, encargada de presentar el asistente al usuario y saludarle.

In [19]:
# Función para generar y reproducir el audio
async def generar_voz(texto):
    # Elegimos la voz: 'es-ES-ElviraNeural' (Mujer, España, muy clara)
    # O 'es-ES-AlvaroNeural' si prefieres hombre.
    VOICE = "es-ES-ElviraNeural"
    OUTPUT_FILE = "respuesta_asistente.mp3"

    communicate = edge_tts.Communicate(texto, VOICE, rate="-10%") # rate="-10%" si la quieres más lenta
    await communicate.save(OUTPUT_FILE)

    # Reproducir en Colab
    display(Audio(OUTPUT_FILE, autoplay=True))

# Forma de ejecutar la función:
# await generar_voz(respuesta_texto)


In [7]:
# Código JavaScript para grabar audio desde el navegador
RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def grabar_audio(segundos=5):
    print(f"Escuchando durante {segundos} segundos...")
    output.eval_js(RECORD_JS)
    audio_b64 = output.eval_js(f"record({segundos*1000})")
    audio_bytes = b64decode(audio_b64.split(',')[1])
    with open("audio_usuario.wav", "wb") as f:
        f.write(audio_bytes)
    return "audio_usuario.wav"

In [65]:
async def presentacion(memoria):
    if memoria.get("nombre") is None:
        texto_bienvenida = "¡Hola! Soy tu Asistente de Salud. Como es nuestra primera vez hablando, no sé nada de ti. Para poder ayudarte mejor... ¿cómo te llamas?"

        print(texto_bienvenida)
        await generar_voz(texto_bienvenida)

        # Pausa para que termine de hablar antes de grabar
        await asyncio.sleep(13)

        # Grabamos al usuario
        archivo_wav = grabar_audio(segundos=5)

        # 1. Transcribimos el audio con Whisper
        resultado = model_whisper.transcribe(archivo_wav, language="es")
        texto_bruto = resultado["text"].strip()
        print(f"Has dicho: '{texto_bruto}'")

        # 2. PROCESAMOS EL NOMBRE CON QWEN
        print("Procesando tu nombre...")

        # Preparamos el prompt estricto para extraer solo el nombre
        mensajes = [
            {"role": "system", "content": "Eres un asistente experto en extracción de entidades. Tu única tarea es leer una frase y devolver ÚNICAMENTE el nombre propio de la persona que se presenta. No añadas puntos, ni saludos, ni explicaciones. Solo la palabra del nombre."},
            {"role": "user", "content": f"Extrae el nombre de esta frase: '{texto_bruto}'"}
        ]

        # Tokenizamos y mandamos al modelo
        texto_prompt = tokenizer_texto.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer_texto([texto_prompt], return_tensors="pt").to(model_texto.device)

        # Generamos la respuesta con temperature baja para mayor precisión
        outputs = model_texto.generate(**inputs, max_new_tokens=10, temperature=0.1)
        nombre_limpio = tokenizer_texto.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        print(f"Nombre extraído: '{nombre_limpio}'")
        # ========================================================

        #Guardamos el nombre limpio en memoria
        memoria["nombre"] = nombre_limpio
        guardar_memoria(memoria)

        texto_confirmacion = f"¡Perfecto, {memoria['nombre']}! He creado tu perfil. Ya podemos empezar con las consultas."
        print(f"\n {texto_confirmacion}")
        await generar_voz(texto_confirmacion)

        # RETORNAMOS LA MEMORIA ACTUALIZADA
        return memoria

    else:
        texto_retorno = f"¡Hola de nuevo, {memoria['nombre']}! ¿Qué hacemos hoy?"
        print(f"{texto_retorno}")
        await generar_voz(texto_retorno)
        return memoria


In [48]:
# Ejecutamos la función
memoria = cargar_memoria()
memoria = await presentacion(memoria)

¡Hola! Soy tu Asistente de Salud. Como es nuestra primera vez hablando, no sé nada de ti. Para poder ayudarte mejor... ¿cómo te llamas?


Escuchando durante 5 segundos...
Has dicho: 'Me llamo Ana.'
Procesando tu nombre...
Nombre extraído: 'Ana'

 ¡Perfecto, Ana! He creado tu perfil. Ya podemos empezar con las consultas.


In [49]:
print(memoria)

{'nombre': 'Ana', 'medicinas': [], 'ultimas_adiciones': []}


#### <a id='toc3_1_2'></a>[2.1.3. Con memoria previa: saludo breve](#toc0_)
Como hemos visto en la función combinada del apartado anterior, si la función detecta que el campo "nombre" ya contiene información, el asistente se salta la fase de recabación de datos. En su lugar, ejecuta directamente un saludo de bienvenida breve y personalizado, confirmándole al usuario que su perfil y su historial médico ya están cargados en memoria y listos para usarse.

In [50]:
# Ejecutamos la función
memoria = cargar_memoria()
memoria = await presentacion(memoria)

¡Hola de nuevo, Ana! ¿Qué hacemos hoy?


## <a id='toc4'></a>[3. Menú de opciones](#toc0_)

Para que el usuario pueda interactuar con las distintas funcionalidades desarrolladas en la Fase B, necesitamos crear una interfaz de texto clara y estructurada.

La función `mostrar_menu()` imprimirá por pantalla un listado con las capacidades de nuestro Asistente (procesamiento de imágenes, gestión de la base de datos, transcripción y síntesis de voz, y lectura de documentos). Utilizaremos un `input` para que el usuario pueda teclear el número correspondiente a la opción que desea utilizar, y la función devolverá dicho valor para que el bucle principal (que programaremos en el siguiente apartado) lo procese.

In [54]:
async def mostrar_menu_voz():
    print("\n" + " · "*10)
    print("       MENÚ PRINCIPAL 🩺")
    print(" · "*10)

    # Texto para pantalla y para la voz
    menu_texto = (
        "Tienes las siguientes opciones:\n"
        "1. Registrar medicamentos mediante imágenes\n"
        "2. Modificar medicamentos\n"
        "3. Ver la tabla resumen de tus tratamientos\n"
        "4. Hacer preguntas por voz\n"
        "5. Usar el lector de documentos\n"
        "6. Ajustes\n"
        "7. Salir del asistente"
    )
    print(f"[Asistente]: {menu_texto}")
    await generar_voz(menu_texto)

    # Esperar unos segundos para que termine de hablar (ajustado a ~16 segundos porque el menú es largo)
    await asyncio.sleep(27)

    # Escuchar respuesta
    archivo_audio = grabar_audio(segundos=5)
    print("Transcribiendo...")
    resultado = model_whisper.transcribe(archivo_audio, language="es")
    eleccion_texto = resultado["text"].strip().lower()
    print(f"Has dicho: '{eleccion_texto}'")

    # Convertir respuesta a número de opción (1 al 7)
    prompt_normalizar = f"""
    El usuario ha dicho: "{eleccion_texto}"

    Devuelve solo un número del 1 al 7 que corresponde a la opción correcta:
    1 → Registrar medicamentos mediante imágenes (o añadir receta, subir foto...)
    2 → Modificar medicamentos (o cambiar, borrar...)
    3 → Ver Tabla Resumen de los tratamientos (o ver historial, ver mis pastillas...)
    4 → Hacer preguntas por voz (o preguntar algo, hablar contigo...)
    5 → Lector de documentos (o leer PDF, leer informe...)
    6 → Ajustes (o cambiar configuración, cambiar nombre...)
    7 → Salir (o apagar, adiós, terminar...)

    Responde SOLO con el número, nada más.
    """

    # Preparar input para Qwen de texto
    messages = [
        {"role": "system", "content": "Eres un asistente que convierte la intención del usuario en un número de opción del 1 al 6."},
        {"role": "user", "content": prompt_normalizar}
    ]

    text_input = tokenizer_texto.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer_texto(text_input, return_tensors="pt").to(model_texto.device)

    print("Interpretando tu elección...")
    with torch.no_grad():
        outputs = model_texto.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None

        )

    # Extraer número
    respuesta_completa = tokenizer_texto.decode(outputs[0], skip_special_tokens=True).strip()

    match = re.search(r'assistant\s*(.*)', respuesta_completa, re.IGNORECASE | re.DOTALL)
    if match:
        respuesta_modelo = match.group(1).strip()
    else:
        respuesta_modelo = respuesta_completa

    # Buscar el primer dígito del 1 al 7
    match_num = re.search(r'\b([1-7])\b', respuesta_modelo)

    if match_num:
        opcion = int(match_num.group(1))
        return opcion
    else:
        # Si no detecta ninguna opción válida, avisa y reinicia
        error_msg = "No he logrado entender qué opción quieres. Vamos a intentarlo de nuevo."
        print(f"⚠️ {error_msg}")
        await generar_voz(error_msg)
        await asyncio.sleep(5)
        return await mostrar_menu_voz() # Llamada recursiva

In [55]:
opcion = await mostrar_menu_voz()


 ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
       MENÚ PRINCIPAL 🩺
 ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
[Asistente]: Tienes las siguientes opciones:
1. Registrar medicamentos mediante imágenes
2. Modificar medicamentos
3. Ver la tabla resumen de tus tratamientos
4. Hacer preguntas por voz
5. Usar el lector de documentos
6. Ajustes
7. Salir del asistente


Escuchando durante 5 segundos...
Transcribiendo...
Has dicho: 'quiero salir del asistente.'
Interpretando tu elección...


In [56]:
print(opcion)

7


In [38]:
opcion = await mostrar_menu_voz()
print(opcion)


 ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
     🩺 MENÚ PRINCIPAL 🩺
 ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
[Asistente]: Tienes las siguientes opciones:
1. Registrar medicamentos mediante imágenes
2. Modificar medicamentos
3. Ver la tabla resumen de tus tratamientos
4. Hacer preguntas por voz
5. Usar el lector de documentos
6. Salir del asistente


Escuchando durante 5 segundos...
Transcribiendo...
Has dicho: 'pero cambiar un medicamento.'
Interpretando tu elección...
2


## <a id='toc5'></a>[4. Cohesión: función global](#toc0_)

Una vez que tenemos preparadas las funciones para gestionar el perfil del usuario (`cargar_memoria` y la `presentacion_multimodal`) y la interfaz de opciones (`mostrar_menu_voz`), es el momento de orquestar todo el sistema.

Para ello, crearemos la función `iniciar_asistente()`. Esta función actuará como el cerebro o "bucle principal" (Main Loop) de nuestra aplicación. Su funcionamiento será el siguiente:
1. Cargará la memoria inicial.
2. Ejecutará el protocolo de bienvenida (saludo o registro, dependiendo de si el usuario es nuevo).
3. Entrará en un bucle `while True` donde, de forma iterativa, mostrará el menú por voz, escuchará la decisión del usuario y derivará la ejecución a la funcionalidad correspondiente de la Fase B.
4. El bucle se mantendrá activo de forma ininterrumpida hasta que el usuario elija explícitamente la opción de "Salir" (Opción 6).

In [68]:
async def iniciar_asistente():
    print(" · "*16)
    print("   🚀 INICIANDO SISTEMA MULTIMODAL DE SALUD...")
    print(" · "*16)

    # 1. Cargar la memoria base desde el archivo JSON
    memoria = cargar_memoria()

    # 2. Presentación (Aquí actualizamos la memoria con el nombre si es nuevo)
    memoria = await presentacion(memoria)
    await asyncio.sleep(3)

    # 3. Bucle del asistente, que acabará cuando el usuario decida
    while True:
        # Llamamos al menú por voz (solo nos devuelve el número del 1 al 6)
        opcion = await mostrar_menu_voz()

        # ==========================================
        # 4. RUTA HACIA LA FASE B
        # ==========================================
        if opcion == 1:
            print("\n💊 [Opción 1] - Registrar medicamentos mediante imágenes")
            # Aquí llamaremos a tu función de analizar recetas con Qwen2-VL
            # memoria = await procesar_receta_imagen(memoria)
            print("... (Conexión con Fase B pendiente) ...")
            await asyncio.sleep(2)

        elif opcion == 2:
            print("\n✏️ [Opción 2] - Modificar medicamentos")
            # Aquí ya tienes tu función ejecutar_opcion_2_menu
            # memoria = await ejecutar_opcion_2_menu(memoria)
            print("... (Conexión con Fase B pendiente) ...")
            await asyncio.sleep(2)

        elif opcion == 3:
            print("\n📋 [Opción 3] - Ver Tabla Resumen de los tratamientos")
            # Aquí mostraremos el HTML/Dataframe del historial
            # mostrar_tabla(memoria)
            print("... (Conexión con Fase B pendiente) ...")
            await asyncio.sleep(2)

        elif opcion == 4:
            print("\n🎙️ [Opción 4] - Preguntas por voz")
            # Integración para hacer preguntas generales sobre las medicinas
            # await responder_dudas_voz(memoria)
            print("... (Conexión con Fase B pendiente) ...")
            await asyncio.sleep(2)

        elif opcion == 5:
            print("\n📄 [Opción 5] - Lector de documentos")
            # Lector de informes PDF / Documentos médicos
            # await leer_informe_medico(memoria)
            print("... (Conexión con Fase B pendiente) ...")
            await asyncio.sleep(2)

        # ==========================================
        # 5. AJUSTES
        # ==========================================
        elif opcion == 6:
            print(" · "*10)
            print("         AJUSTES 🔧")
            print(" · "*10)

            await asyncio.sleep(2)

        # ==========================================
        # 6. CERRAR SESIÓN
        # ==========================================

        elif opcion == 7:
            # Despedida y salimos del bucle
            despedida = f"¡Hasta pronto, {memoria['nombre']}! Cuídate mucho y recuerda seguir tus tratamientos."
            print(f"\n👋 {despedida}")
            await generar_voz(despedida)
            break # El break es vital para salir del "while True" y apagar el asistente

In [69]:
# ¡Que comience la magia!
await iniciar_asistente()

 ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
   🚀 INICIANDO SISTEMA MULTIMODAL DE SALUD...
 ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
¡Hola de nuevo, Ana! ¿Qué hacemos hoy?



 ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
       MENÚ PRINCIPAL 🩺
 ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
[Asistente]: Tienes las siguientes opciones:
1. Registrar medicamentos mediante imágenes
2. Modificar medicamentos
3. Ver la tabla resumen de tus tratamientos
4. Hacer preguntas por voz
5. Usar el lector de documentos
6. Ajustes
7. Salir del asistente


Escuchando durante 5 segundos...
Transcribiendo...
Has dicho: 'y voy a ajustar cosas'
Interpretando tu elección...

📄 [Opción 5] - Lector de documentos
... (Conexión con Fase B pendiente) ...

 ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
       MENÚ PRINCIPAL 🩺
 ·  ·  ·  ·  ·  ·  ·  ·  ·  · 
[Asistente]: Tienes las siguientes opciones:
1. Registrar medicamentos mediante imágenes
2. Modificar medicamentos
3. Ver la tabla resumen de tus tratamientos
4. Hacer preguntas por voz
5. Usar el lector de documentos
6. Ajustes
7. Salir del asistente


Escuchando durante 5 segundos...
Transcribiendo...
Has dicho: 'quiero salir del asistente.'
Interpretando tu elección...

👋 ¡Hasta pronto, Ana! Cuídate mucho y recuerda seguir tus tratamientos.
